In [1]:
#conect with drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["GITHUB_TOKEN"] = "xx"
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"] = "xx"

In [3]:
#!sed -i 's/\r$//' /content/drive/MyDrive/apai/fetch_src.sh  # Windows users
!bash /content/drive/MyDrive/apai/fetch_src.sh

 fetch_src.sh — Download /src from GitHub

Checking environment...
GITHUB_TOKEN set
Fetching file list from GitHub API...
Found 19 file(s) under /src/
src/cl_strategies/ewc.py -> /content/src/cl_strategies/ewc.py
src/cl_strategies/naive.py -> /content/src/cl_strategies/naive.py
src/cl_strategies/reharsal.py -> /content/src/cl_strategies/reharsal.py
src/config/config.py -> /content/src/config/config.py
src/data/dataset.py -> /content/src/data/dataset.py
src/data/preprocessing.py -> /content/src/data/preprocessing.py
src/data/study_dataset.py -> /content/src/data/study_dataset.py
src/fine_tune/trainer.py -> /content/src/fine_tune/trainer.py
src/fine_tune/visualizer.py -> /content/src/fine_tune/visualizer.py
src/imports.py -> /content/src/imports.py
src/meta_learning/buffers.py -> /content/src/meta_learning/buffers.py
src/meta_learning/data.py -> /content/src/meta_learning/data.py
src/meta_learning/embeddings.py -> /content/src/meta_learning/embeddings.py
src/meta_learning/models.py -> /c

In [4]:
#!sed -i 's/\r$//' /content/drive/MyDrive/apai/setup_colab.sh    # Windows users
!bash /content/drive/MyDrive/apai/setup_colab.sh


UCF101 Project — Colab Setup Script

Checking environment...
KAGGLE_USERNAME set
KAGGLE_KEY set
config.py found

Installing kagglehub...
Done.

Using Colab cache for faster access to the 'ucf101-action-recognition' dataset.
Dataset path: /kaggle/input/ucf101-action-recognition

Detecting path and updating config.py...
Found in Kaggle input: /kaggle/input/ucf101-action-recognition/
DATASET_ROOT = '/kaggle/input/ucf101-action-recognition/'
OUTPUT_ROOT  = '/kaggle/working/ucf101-processed/'

Setup complete!
config.py updated on Drive.
Restarting kernel in 5 seconds...
Dataset cache stays on disk.
Run Cell 3 after restart to verify.
Restarting in 5...
Restarting in 4...
Restarting in 3...
Restarting in 2...
Restarting in 1...


In [5]:
# imports
%run /content/src/imports.py

Using device: cuda


## **Paths & Hyperparameters**

In [6]:
# preprocessing roots
TASK0_ROOT           = "./UCF101/processed_task0"
EXEMPLAR_ROOT        = "./UCF101/processed_exemplars"
SUBSET1_ROOT         = "./UCF101/processed_subset1"
LIMITED_SUBSET1_ROOT = "./UCF101/processed_limited_subset1"
SUBSET2_ROOT         = "./UCF101/processed_subset2"
LIMITED_SUBSET2_ROOT = "./UCF101/processed_limited_subset2"

# embedding roots
TASK0_EMB_ROOT    = "./UCF101/emb_task0"
EXEMPLAR_EMB_ROOT = "./UCF101/emb_exemplars"
SUBSET1_EMB_ROOT  = "./UCF101/emb_subset1"
LIM_SUB1_EMB_ROOT = "./UCF101/emb_limited_subset1"
SUBSET2_EMB_ROOT  = "./UCF101/emb_subset2"
LIM_SUB2_EMB_ROOT = "./UCF101/emb_limited_subset2"

ALL_EMB_ROOTS = [
    TASK0_EMB_ROOT, EXEMPLAR_EMB_ROOT,
    SUBSET1_EMB_ROOT, LIM_SUB1_EMB_ROOT,
    SUBSET2_EMB_ROOT, LIM_SUB2_EMB_ROOT,
]


In [ ]:
# training hyperparameters
LIMIT = 5
NEW_REPEAT = 2

REP_N_WAY = 3
REP_K_SUPPORT = 2
REP_K_QUERY = 1

T1_INNER_STEPS = 3
T1_EPOCHS = 5

T2_INNER_STEPS = 3
T2_EPOCHS = 5

REH_LR = 1e-4
REP_LSTM_LR = 1e-4
REP_FC_LR = 1e-3
REP_INNER_LR = 1e-2
REP_EPSILON = 0.05
LAMBDA_DISTILL = 0.3
LAMBDA_KD_REPTILE = 0.1
LAMBDA_QRY = 0.5


# class indexing
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx = {cls: i for i, cls in enumerate(ALL_CLASSES_LIST)}

N0 = len(cfg.SELECTED_CLASSES)
N1 = len(cfg.TASK_1)
N2 = len(cfg.TASK_2)
N_TOTAL = N0 + N1 + N2

## **Preprocess**

In [11]:
# preprocess just the val,test on the old classes
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=TASK0_ROOT, splits=["val"])
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=TASK0_ROOT, splits=["test"])          

# old classes trian limited for reply buffer
preprocess_dataset(target_classes=cfg.SELECTED_CLASSES,
                   output_root=EXEMPLAR_ROOT, max_samples=LIMIT, splits=["train"])

# task 1 few-shot
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=LIMITED_SUBSET1_ROOT, max_samples=LIMIT, splits=["train"])
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=SUBSET1_ROOT, splits=["val"])
preprocess_dataset(target_classes=cfg.TASK_1,
                   output_root=SUBSET1_ROOT, splits=["test"])

# task 2 few-shot
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=LIMITED_SUBSET2_ROOT, max_samples=LIMIT, splits=["train"])
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=SUBSET2_ROOT, splits=["val"])
preprocess_dataset(target_classes=cfg.TASK_2,
                   output_root=SUBSET2_ROOT, splits=["test"])


--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_task0
Classes: 10 | Limit: Full

Processing split: val



Done! Dataset ready at: ./UCF101/processed_task0

--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_task0
Classes: 10 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_task0

--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_exemplars
Classes: 10 | Limit: 5

Processing split: train



Done! Dataset ready at: ./UCF101/processed_exemplars

--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_limited_subset1
Classes: 3 | Limit: 5

Processing split: train



Done! Dataset ready at: ./UCF101/processed_limited_subset1

--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_subset1
Classes: 3 | Limit: Full

Processing split: val



Done! Dataset ready at: ./UCF101/processed_subset1

--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_subset1
Classes: 3 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_subset1

--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_limited_subset2
Classes: 3 | Limit: 5

Processing split: train



Done! Dataset ready at: ./UCF101/processed_limited_subset2

--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_subset2
Classes: 3 | Limit: Full

Processing split: val



Done! Dataset ready at: ./UCF101/processed_subset2

--- Preprocessing UCF101 ---
From: /kaggle/input/ucf101-action-recognition/
To  : ./UCF101/processed_subset2
Classes: 3 | Limit: Full

Processing split: test



Done! Dataset ready at: ./UCF101/processed_subset2


## **Upload Model**

In [12]:
model = ResNet50LSTM(num_classes=N0).to(device)
model.load_state_dict(torch.load(cfg.RESNET50_PATH, map_location=device))

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 176MB/s]


<All keys matched successfully>

In [13]:
model_embeddings = ResNet50Trunk(model).to(device)
model_embeddings.eval() # evaluation
del model  # keep only the backbone

## **Exctract embeddings**

In [14]:
extraction_plan = [
    (TASK0_ROOT,           TASK0_EMB_ROOT,    "Task 0        -> val&test"),
    (EXEMPLAR_ROOT,        EXEMPLAR_EMB_ROOT, f"Exemplars     -> {LIMIT}/cls train  <- replay buffer"),
    (LIMITED_SUBSET1_ROOT, LIM_SUB1_EMB_ROOT, f"Task 1 train  -> {LIMIT}/cls (few-shot)"),
    (SUBSET1_ROOT,         SUBSET1_EMB_ROOT,  "Task 1        -> val & test"),
    (LIMITED_SUBSET2_ROOT, LIM_SUB2_EMB_ROOT, f"Task 2 train  -> {LIMIT}/cls (few-shot)"),
    (SUBSET2_ROOT,         SUBSET2_EMB_ROOT,  "Task 2        -> val & test"),
]

# delete existing dirs
for d in ALL_EMB_ROOTS:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f"  Deleted stale dir: {d}")

for clip_root, emb_root, label in extraction_plan:
    print(f"  -> {label}")
    extract_embeddings(clip_root, emb_root, model_embeddings, device)
    torch.cuda.empty_cache()

  -> Task 0        -> val&test


  -> Exemplars     -> 5/cls train  <- replay buffer


  -> Task 1 train  -> 5/cls (few-shot)


  -> Task 1        -> val & test


  -> Task 2 train  -> 5/cls (few-shot)


  -> Task 2        -> val & test


## **Loaders**

In [ ]:
# DATASETS
# Task 0: val & test
task0_val_ds  = UCF101Embeddings(f"{TASK0_EMB_ROOT}/val",  class_to_idx)
task0_test_ds = UCF101Embeddings(f"{TASK0_EMB_ROOT}/test", class_to_idx)

# exemplars limit classes from T0 train
exemplar_train_ds = UCF101Embeddings(f"{EXEMPLAR_EMB_ROOT}/train", class_to_idx)

# Task 1: few-shot
task1_train_ds = UCF101Embeddings(f"{LIM_SUB1_EMB_ROOT}/train", class_to_idx)
task1_val_ds   = UCF101Embeddings(f"{SUBSET1_EMB_ROOT}/val",    class_to_idx)
task1_test_ds  = UCF101Embeddings(f"{SUBSET1_EMB_ROOT}/test",   class_to_idx)

# Task 2: few-shot
task2_train_ds = UCF101Embeddings(f"{LIM_SUB2_EMB_ROOT}/train", class_to_idx)
task2_val_ds   = UCF101Embeddings(f"{SUBSET2_EMB_ROOT}/val",    class_to_idx)
task2_test_ds  = UCF101Embeddings(f"{SUBSET2_EMB_ROOT}/test",   class_to_idx)


# DATALOADERS
task0_val_loader      = _dl(task0_val_ds)
task0_test_loader     = _dl(task0_test_ds)
exemplar_train_loader = _dl(exemplar_train_ds)   # shuffle=False

task1_train_loader = _dl(task1_train_ds, shuffle=True)
task1_val_loader   = _dl(task1_val_ds)
task1_test_loader  = _dl(task1_test_ds)

task2_train_loader = _dl(task2_train_ds, shuffle=True)
task2_val_loader   = _dl(task2_val_ds)
task2_test_loader  = _dl(task2_test_ds)

# combined loaders
combined_val_t1_loader = _dl(ConcatDataset([task0_val_ds, task1_val_ds]), shuffle=True)
combined_val_t2_loader = _dl(ConcatDataset([task0_val_ds, task1_val_ds, task2_val_ds]), shuffle=True)

combined_test_t1_loader = _dl(ConcatDataset([task0_test_ds, task1_test_ds]))
combined_test_t2_loader = _dl(ConcatDataset([task0_test_ds, task1_test_ds, task2_test_ds]))

combined_exemplar_t2_ds = ConcatDataset([exemplar_train_ds, task1_train_ds])
combined_exemplar_t2_loader = _dl(combined_exemplar_t2_ds)

## **Buffers**

In [16]:
# class names in index order
num_classes = len(class_to_idx)
class_names = [None] * num_classes
for cls_name, idx in class_to_idx.items():
    class_names[idx] = cls_name


In [ ]:
# BUFFER POPULATION
# Task 1 buffers
replay_buf_t1 = ReplayBuffer(max_size=1000)
replay_buf_t1_kd = ReplayBuffer(max_size=1000)

replay_buf_t1.add_from_loader(exemplar_train_loader, max_per_class=LIMIT)
replay_buf_t1_kd.add_from_loader(exemplar_train_loader, max_per_class=LIMIT)

episode_buffer_t1 = EpisodeBuffer()
episode_buffer_t1_kd = EpisodeBuffer()

episode_buffer_t1.add_from_loader(exemplar_train_loader, mark_new=False, max_per_class=LIMIT)
episode_buffer_t1_kd.add_from_loader(exemplar_train_loader, mark_new=False, max_per_class=LIMIT)

n_way_eff_t1 = min(REP_N_WAY, len(episode_buffer_t1.available_classes()))
n_way_eff_t1_kd = min(REP_N_WAY, len(episode_buffer_t1_kd.available_classes()))

print(f"[T1 Replay] {len(replay_buf_t1)} samples")
print(f"[T1 Episode] {len(episode_buffer_t1)} samples | n_way_eff={n_way_eff_t1}")

# Task 2 buffers
replay_buf_t2 = ReplayBuffer(max_size=1000)
replay_buf_t2_kd = ReplayBuffer(max_size=1000)

replay_buf_t2.add_from_loader(exemplar_train_loader, max_per_class=LIMIT)
replay_buf_t2.add_from_loader(task1_train_loader, max_per_class=LIMIT)

replay_buf_t2_kd.add_from_loader(exemplar_train_loader, max_per_class=LIMIT)
replay_buf_t2_kd.add_from_loader(task1_train_loader, max_per_class=LIMIT)

episode_buffer_t2 = EpisodeBuffer()
episode_buffer_t2_kd = EpisodeBuffer()

episode_buffer_t2.add_from_loader(exemplar_train_loader, mark_new=False, max_per_class=LIMIT)
episode_buffer_t2.add_from_loader(task1_train_loader, mark_new=False, max_per_class=LIMIT)
episode_buffer_t2.add_from_loader(task2_train_loader, mark_new=True, max_per_class=LIMIT)

episode_buffer_t2_kd.add_from_loader(exemplar_train_loader, mark_new=False, max_per_class=LIMIT)
episode_buffer_t2_kd.add_from_loader(task1_train_loader, mark_new=False, max_per_class=LIMIT)
episode_buffer_t2_kd.add_from_loader(task2_train_loader, mark_new=True, max_per_class=LIMIT)

n_way_eff_t2 = min(REP_N_WAY, len(episode_buffer_t2.available_classes()))
n_way_eff_t2_kd = min(REP_N_WAY, len(episode_buffer_t2_kd.available_classes()))

print(f"[T2 Replay] {len(replay_buf_t2)} samples")
print(f"[T2 Episode] {len(episode_buffer_t2)} samples | new_classes={episode_buffer_t2.new_classes} | "
      f"n_way_eff={n_way_eff_t2}")

In [15]:
buffer_comparison = {
    "Task 1 Replay": replay_buf_t1,
    "Task 1 KD Replay": replay_buf_t1_kd,
    "Task 1 Episode": episode_buffer_t1,
    "Task 1 KD Episode": episode_buffer_t1_kd,
    "Task 2 Replay": replay_buf_t2,
    "Task 2 KD Replay": replay_buf_t2_kd,
    "Task 2 Episode": episode_buffer_t2,
    "Task 2 KD Episode": episode_buffer_t2_kd
}

fig = plot_buffer_composition(buffer_comparison, class_names=class_names)

fig.show()

## **Evaluate & Record ResNet50LSTM**

In [16]:
model_embeddings = fresh_model(num_classes=N0 + N1, cfg=cfg, device=device, lstm_hidden=256)
model_embeddings.eval()

W0_base_t1 = {k: v.cpu().clone() for k, v in model_embeddings.state_dict().items()}

record(
    name="Base ResNet50LSTM",
    model=model_embeddings,
    results_dict=results_t1,
    old_loader=task0_test_loader,
    new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader,
    embed_loader=combined_test_t1_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1)],
    W0=W0_base_t1,
)

model_embeddings = fresh_model(num_classes=N_TOTAL, cfg=cfg, device=device, lstm_hidden=256)
model_embeddings.eval()

W0_base_t2 = {k: v.cpu().clone() for k, v in model_embeddings.state_dict().items()}

record(
    name="Base ResNet50LSTM",
    model=model_embeddings,
    results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader,
    embed_loader=combined_test_t2_loader,
    device=device,
    class_names=[str(i) for i in range(N_TOTAL)],
    W0=W0_base_t2,
)

# cleanup
del model_embeddings
del W0_base_t1
del W0_base_t2
torch.cuda.empty_cache()

  [Base ResNet50LSTM] Old: 0.0234 | New: 0.0755 | All: 0.0357
  [Base ResNet50LSTM] Old: 0.1161 | New: 0.0909 | All: 0.1111


## **Rehersial**

- No Knowladge Distilation

In [ ]:
m_reh_t1 = build_student_from_task0(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    num_old_classes=N0
)

optimizer_reh_t1 = make_rehearsal_optimizer(m_reh_t1, lr=REH_LR)
W0_reh_t1 = snapshot_weights(m_reh_t1)

m_reh_t1 = train_rehearsal(
    model=m_reh_t1,
    teacher=None,
    train_loader=task1_train_loader,
    val_loader=combined_val_t1_loader,
    replay_buffer=replay_buf_t1,
    optimizer=optimizer_reh_t1,
    device=device,
    num_old_classes=N0,
    lambda_distill=0.0,
    epochs=T1_EPOCHS,
    new_repeat=NEW_REPEAT,
    kd=False,
)

record(
    name="Rehearsal T1 (no-KD)",
    model=m_reh_t1,
    results_dict=results_t1,
    old_loader=task0_test_loader,
    new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader,
    embed_loader=combined_test_t1_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1)],
    W0=W0_reh_t1,
)

[Rehearsal no-KD] Epoch 1/5 | Loss: 16.3145 | ValAcc: 0.1767
[Rehearsal no-KD] Epoch 2/5 | Loss: 10.0355 | ValAcc: 0.2372
[Rehearsal no-KD] Epoch 3/5 | Loss: 7.0823 | ValAcc: 0.2419
[Rehearsal no-KD] Epoch 4/5 | Loss: 5.0643 | ValAcc: 0.2605
[Rehearsal no-KD] Epoch 5/5 | Loss: 3.7270 | ValAcc: 0.2465
  [Rehearsal no-KD] Old: 0.0994 | New: 0.8113 | All: 0.2679


In [ ]:
plot_confusion({k: results_t1[k] for k in ["Rehearsal T1 (no-KD)"]}, task_label="T1", global_class_names=class_names).show()

In [ ]:
#plot_umap({k: results_t1[k] for k in ["Rehearsal no-KD"]}, task_label="T1", global_class_names=class_names).show()
#plot_umap(results_t1, task_label="T1", global_class_names=class_names).show()

In [ ]:
#plot_weight_delta({k: results_t1[k] for k in ["Rehearsal no-KD"]}, task_label="T1").show()

## **With Knowladge Distilation**

In [ ]:


teacher_base = make_frozen_teacher_from_checkpoint(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    checkpoint_path=cfg.RESNET50_PATH
)

print("Teacher training mode:", teacher_base.training)
print("Teacher has trainable params:", any(p.requires_grad for p in teacher_base.parameters()))


m_reh_kd_t1 = build_student_from_task0(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    num_old_classes=N0
)

optimizer_reh_kd_t1 = make_rehearsal_optimizer(m_reh_kd_t1, lr=REH_LR)
W0_reh_kd_t1 = snapshot_weights(m_reh_kd_t1)

m_reh_kd_t1 = train_rehearsal(
    model=m_reh_kd_t1,
    teacher=teacher_base,
    train_loader=task1_train_loader,
    val_loader=combined_val_t1_loader,
    replay_buffer=replay_buf_t1_kd,
    optimizer=optimizer_reh_kd_t1,
    device=device,
    num_old_classes=N0,
    lambda_distill=LAMBDA_DISTILL,
    epochs=T1_EPOCHS,
    new_repeat=NEW_REPEAT,
    kd=True,
)

record(
    name="Rehearsal T1 (KD)",
    model=m_reh_kd_t1,
    results_dict=results_t1,
    old_loader=task0_test_loader,
    new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader,
    embed_loader=combined_test_t1_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1)],
    W0=W0_reh_kd_t1,
)

[Rehearsal KD] Epoch 1/5 | Loss: 18.4951 | ValAcc: 0.0419
[Rehearsal KD] Epoch 2/5 | Loss: 18.3289 | ValAcc: 0.0419
[Rehearsal KD] Epoch 3/5 | Loss: 18.0209 | ValAcc: 0.0419
[Rehearsal KD] Epoch 4/5 | Loss: 17.9171 | ValAcc: 0.0419
[Rehearsal KD] Epoch 5/5 | Loss: 17.8281 | ValAcc: 0.0419
  [Rehearsal T1 (KD, best)] Old: 0.0468 | New: 0.0755 | All: 0.0536


In [ ]:
#fig = plot_accuracy(results_t1, task_label="T1")
#fig.show()

In [ ]:
#plot_confusion(results_t1, task_label="T1", global_class_names=class_names).show()

In [ ]:
#plot_umap(results_t1, task_label="T1", global_class_names=class_names).show()

In [ ]:
#plot_weight_delta(results_t1, task_label="T1").show()

## **REPTILE**

In [ ]:



# =========================================================
# TASK 1 — REPTILE no-KD
# =========================================================

m_rep_t1 = build_student_from_task0(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    num_old_classes=N0
)

W0_rep_t1 = snapshot_weights(m_rep_t1)

m_rep_t1 = train_reptile(
    model=m_rep_t1,
    episode_buffer=episode_buffer_t1,
    train_loader=task1_train_loader,
    val_loader=combined_val_t1_loader,
    device=device,
    teacher=None,
    num_old_classes=N0,
    n_way=n_way_eff_t1,
    k_support=REP_K_SUPPORT,
    k_query=REP_K_QUERY,
    inner_lr=REP_INNER_LR,
    lstm_lr=REP_LSTM_LR,
    fc_lr=REP_FC_LR,
    inner_steps=T1_INNER_STEPS,
    epsilon=REP_EPSILON,
    new_repeat=NEW_REPEAT,
    epochs=T1_EPOCHS,
    lambda_kd=0.0,
    lambda_qry=LAMBDA_QRY,
    kd=False,
    tag="Reptile-T1-noKD",
)

record(
    name="Reptile T1 (no-KD)",
    model=m_rep_t1,
    results_dict=results_t1,
    old_loader=task0_test_loader,
    new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader,
    embed_loader=combined_test_t1_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1)],
    W0=W0_rep_t1,
)

[Reptile-T1-KD-Manual] Epoch 1 | Val Acc: 0.8047
[Reptile-T1-KD-Manual] Epoch 2 | Val Acc: 0.8558
[Reptile-T1-KD-Manual] Epoch 3 | Val Acc: 0.8186
[Reptile-T1-KD-Manual] Epoch 4 | Val Acc: 0.8326
[Reptile-T1-KD-Manual] Epoch 5 | Val Acc: 0.8465
  [Reptile T1 (KD, Manual)] Old: 0.9591 | New: 0.7358 | All: 0.9062


In [34]:
fig = plot_accuracy(results_t1, task_label="T1")
fig.show()

In [ ]:
#plot_confusion(results_t1, task_label="T1", global_class_names=class_names).show()

In [ ]:
#plot_umap(results_t1, task_label="T1", global_class_names=class_names).show()

In [ ]:
#plot_weight_delta(results_t1, task_label="T1").show()

## **With KD**

In [ ]:
m_rep_kd_t1 = build_student_from_task0(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    num_old_classes=N0
)

W0_rep_kd_t1 = snapshot_weights(m_rep_kd_t1)

m_rep_kd_t1 = train_reptile(
    model=m_rep_kd_t1,
    episode_buffer=episode_buffer_t1_kd,
    train_loader=task1_train_loader,
    val_loader=combined_val_t1_loader,
    device=device,
    teacher=teacher_base,
    num_old_classes=N0,
    n_way=n_way_eff_t1_kd,
    k_support=REP_K_SUPPORT,
    k_query=REP_K_QUERY,
    inner_lr=REP_INNER_LR,
    lstm_lr=REP_LSTM_LR,
    fc_lr=REP_FC_LR,
    inner_steps=T1_INNER_STEPS,
    epsilon=REP_EPSILON,
    new_repeat=NEW_REPEAT,
    epochs=T1_EPOCHS,
    lambda_kd=LAMBDA_KD_REPTILE,
    lambda_qry=LAMBDA_QRY,
    kd=True,
    tag="Reptile-T1-KD",
)

record(
    name="Reptile T1 (KD)",
    model=m_rep_kd_t1,
    results_dict=results_t1,
    old_loader=task0_test_loader,
    new_loader=task1_test_loader,
    all_loader=combined_test_t1_loader,
    embed_loader=combined_test_t1_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1)],
    W0=W0_rep_kd_t1,
)

✅ Pesi Task 0 caricati correttamente (Layer filtrati: 4)
[Reptile-T1-KD-Manual-Final] Epoch 1 | Val Acc: 0.8186
[Reptile-T1-KD-Manual-Final] Epoch 2 | Val Acc: 0.8605
[Reptile-T1-KD-Manual-Final] Epoch 3 | Val Acc: 0.8651
[Reptile-T1-KD-Manual-Final] Epoch 4 | Val Acc: 0.8837
[Reptile-T1-KD-Manual-Final] Epoch 5 | Val Acc: 0.8744
  [Reptile T1 (KD, Manual best)] Old: 0.9474 | New: 0.8491 | All: 0.9241


In [ ]:
#fig = plot_accuracy(results_t1, task_label="T1")
#fig.show()

In [ ]:
#plot_confusion(results_t1, task_label="T1", global_class_names=class_names).show()

In [ ]:
#plot_umap(results_t1, task_label="T1", global_class_names=class_names).show()

In [ ]:
#plot_weight_delta(results_t1, task_label="T1").show()

## **Plots**

In [ ]:
#plot_per_class_accuracy(results_t1, task_label="T1", global_class_names=class_names).show()

## **TASK 2**

- rehersial no-kd

In [ ]:
m_reh_t2 = build_student_from_task0(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    num_old_classes=N0 + N1
)

optimizer_reh_t2 = make_rehearsal_optimizer(m_reh_t2, lr=REH_LR)
W0_reh_t2 = snapshot_weights(m_reh_t2)

m_reh_t2 = train_rehearsal(
    model=m_reh_t2,
    teacher=None,
    train_loader=task2_train_loader,
    val_loader=combined_val_t2_loader,
    replay_buffer=replay_buf_t2,
    optimizer=optimizer_reh_t2,
    device=device,
    num_old_classes=N0 + N1,
    lambda_distill=0.0,
    epochs=T2_EPOCHS,
    new_repeat=NEW_REPEAT,
    kd=False,
)

record(
    name="Rehearsal T2 (no-KD)",
    model=m_reh_t2,
    results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader,
    embed_loader=combined_test_t2_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1 + N2)],
    W0=W0_reh_t2,
)

TypeError: tuple indices must be integers or slices, not str

- rehersial kd

In [ ]:

m_reh_kd_t2 = build_student_from_task0(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    num_old_classes=N0 + N1
)

optimizer_reh_kd_t2 = make_rehearsal_optimizer(m_reh_kd_t2, lr=REH_LR)
W0_reh_kd_t2 = snapshot_weights(m_reh_kd_t2)

m_reh_kd_t2 = train_rehearsal(
    model=m_reh_kd_t2,
    teacher=teacher_base,
    train_loader=task2_train_loader,
    val_loader=combined_val_t2_loader,
    replay_buffer=replay_buf_t2_kd,
    optimizer=optimizer_reh_kd_t2,
    device=device,
    num_old_classes=N0 + N1,
    lambda_distill=LAMBDA_DISTILL,
    epochs=T2_EPOCHS,
    new_repeat=NEW_REPEAT,
    kd=True,
)

record(
    name="Rehearsal T2 (KD)",
    model=m_reh_kd_t2,
    results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader,
    embed_loader=combined_test_t2_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1 + N2)],
    W0=W0_reh_kd_t2,
)


[Rehearsal KD] Epoch 1/5 | Loss: 24.8240 | ValAcc: 0.6642
[Rehearsal KD] Epoch 2/5 | Loss: 10.8313 | ValAcc: 0.7090
[Rehearsal KD] Epoch 3/5 | Loss: 9.3053 | ValAcc: 0.7164
[Rehearsal KD] Epoch 4/5 | Loss: 6.9001 | ValAcc: 0.7127
[Rehearsal KD] Epoch 5/5 | Loss: 6.2438 | ValAcc: 0.7313
 Rehearsal T2 (KD, best)              Old:0.7902  New:0.5455  All:0.7419


- Reptile no-kd

In [ ]:
m_rep_t2 = build_student_from_task0(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    num_old_classes=N0 + N1
)

W0_rep_t2 = snapshot_weights(m_rep_t2)

m_rep_t2 = train_reptile(
    model=m_rep_t2,
    episode_buffer=episode_buffer_t2,
    train_loader=task2_train_loader,
    val_loader=combined_val_t2_loader,
    device=device,
    teacher=None,
    num_old_classes=N0 + N1,
    n_way=n_way_eff_t2,
    k_support=REP_K_SUPPORT,
    k_query=REP_K_QUERY,
    inner_lr=REP_INNER_LR,
    lstm_lr=REP_LSTM_LR,
    fc_lr=REP_FC_LR,
    inner_steps=T2_INNER_STEPS,
    epsilon=REP_EPSILON,
    new_repeat=NEW_REPEAT,
    epochs=T2_EPOCHS,
    lambda_kd=0.0,
    lambda_qry=LAMBDA_QRY,
    kd=False,
    tag="Reptile-T2-noKD",
)

record(
    name="Reptile T2 (no-KD)",
    model=m_rep_t2,
    results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader,
    embed_loader=combined_test_t2_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1 + N2)],
    W0=W0_rep_t2,
)

[Reptile-T2-noKD-final] Epoch 1/5 | ValAcc: 0.7500 | QryLoss: 0.0235
[Reptile-T2-noKD-final] Epoch 2/5 | ValAcc: 0.7388 | QryLoss: 0.0097
[Reptile-T2-noKD-final] Epoch 3/5 | ValAcc: 0.7500 | QryLoss: 0.0091
[Reptile-T2-noKD-final] Epoch 4/5 | ValAcc: 0.7425 | QryLoss: 0.0038
[Reptile-T2-noKD-final] Epoch 5/5 | ValAcc: 0.7425 | QryLoss: 0.0033
  [Reptile-T2-noKD-final] Best val acc : 0.7500
 Reptile T2 (no KD, best)             Old:0.8348  New:0.4727  All:0.7634


- reptile kd

In [ ]:
m_rep_kd_t2 = build_student_from_task0(
    num_classes=N_TOTAL,
    cfg=cfg,
    device=device,
    lstm_hidden=256,
    num_old_classes=N0 + N1
)

W0_rep_kd_t2 = snapshot_weights(m_rep_kd_t2)

m_rep_kd_t2 = train_reptile(
    model=m_rep_kd_t2,
    episode_buffer=episode_buffer_t2_kd,
    train_loader=task2_train_loader,
    val_loader=combined_val_t2_loader,
    device=device,
    teacher=teacher_base,
    num_old_classes=N0 + N1,
    n_way=n_way_eff_t2_kd,
    k_support=REP_K_SUPPORT,
    k_query=REP_K_QUERY,
    inner_lr=REP_INNER_LR,
    lstm_lr=REP_LSTM_LR,
    fc_lr=REP_FC_LR,
    inner_steps=T2_INNER_STEPS,
    epsilon=REP_EPSILON,
    new_repeat=NEW_REPEAT,
    epochs=T2_EPOCHS,
    lambda_kd=LAMBDA_KD_REPTILE,
    lambda_qry=LAMBDA_QRY,
    kd=True,
    tag="Reptile-T2-KD",
)

record(
    name="Reptile T2 (KD)",
    model=m_rep_kd_t2,
    results_dict=results_t2,
    old_loader=combined_test_t1_loader,
    new_loader=task2_test_loader,
    all_loader=combined_test_t2_loader,
    embed_loader=combined_test_t2_loader,
    device=device,
    class_names=[str(i) for i in range(N0 + N1 + N2)],
    W0=W0_rep_kd_t2,
)

## **Plots**

In [ ]:

cl_history = {}

for method_name, metrics in results_t1.items():
    cl_history[method_name] = [metrics['all_acc']]

plot_cl_curves(cl_history).show()